# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5a1892700', 'id': 'chatcmpl-DuuNaz2sDbnJP5pMRIEVmzDVXaRth', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f029f-e8bf-7b71-ac41-890476301924-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool

In [3]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [4]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [6]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [7]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [8]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_S571EghxkrGJePTZpNGZs42q', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTTcPI5om6KnsKhGC9zs80GdL3T', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-7a12-7262-8282-e3dfcc01f556-0', tool_cal

In [9]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_S571EghxkrGJePTZpNGZs42q',
  'type': 'tool_call'}]

In [14]:
tool_dict[response.tool_calls[0]['name']].invoke(response.tool_calls[0])  # invoke()

Asia/Seoul (서울) 현재시각 2026-06-26 15:40:51 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:40:51 ', name='get_current_time', tool_call_id='call_S571EghxkrGJePTZpNGZs42q')

In [17]:
tool_call = response.tool_calls[0]
tool_call["name"]

'get_current_time'

In [18]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_S571EghxkrGJePTZpNGZs42q',
  'type': 'tool_call'}]

In [19]:
tool_dict[tool_call["name"]]

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x0000017EE0CEB160>)

In [20]:
tool_dict[tool_call["name"]].invoke(tool_call)

Asia/Seoul (서울) 현재시각 2026-06-26 15:43:23 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:43:23 ', name='get_current_time', tool_call_id='call_S571EghxkrGJePTZpNGZs42q')

In [21]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '서울'}
Asia/Seoul (서울) 현재시각 2026-06-26 15:43:32 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_S571EghxkrGJePTZpNGZs42q', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTTcPI5om6KnsKhGC9zs80GdL3T', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-7a12-7262-8282-e3dfcc01f556-0', tool_cal

In [22]:
llm_with_tools.invoke(messages)

AIMessage(content='서울은 지금 2026년 6월 26일 오후 3시 43분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 189, 'total_tokens': 212, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-Duua1dvhaBOgOjb17xWbXBRq7aWKB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02ab-a9dc-7563-800d-44c7eee709c3-0', usage_metadata={'input_tokens': 189, 'output_tokens': 23, 'total_tokens': 212, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [23]:
messages.append(llm_with_tools.invoke(messages))

In [24]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_S571EghxkrGJePTZpNGZs42q', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTTcPI5om6KnsKhGC9zs80GdL3T', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-7a12-7262-8282-e3dfcc01f556-0', tool_cal

### 주가 예제

In [25]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [26]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [27]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_Ljaw8H2uUttjNFv79CnQnXVt',
  'type': 'tool_call'}]

In [30]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_S571EghxkrGJePTZpNGZs42q', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTTcPI5om6KnsKhGC9zs80GdL3T', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-7a12-7262-8282-e3dfcc01f556-0', tool_cal

In [33]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_Ljaw8H2uUttjNFv79CnQnXVt',
  'type': 'tool_call'}]

In [35]:
!pip install tabulate

In [40]:
#### 이어서 자연어로 모델이 답변하는 것 추가 실습

for tool_call in response.tool_calls:
    out = tool_dict[tool_call['name']].invoke(tool_call)
    messages.append(out)

llm_with_tools.invoke(messages)

AIMessage(content='한 달 전인 2026년 5월 26일 테슬라(TSLA)의 종가는 433.59달러였고, 현재(2026년 6월 25일)의 종가는 375.12달러입니다. \n\n따라서, 테슬라의 주가는 한 달 전에 비해 하락했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 1670, 'total_tokens': 1745, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a151a364ab', 'id': 'chatcmpl-DuupCeIbnrXya7Z9QkdoRRDWxX1SJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02ba-07b1-70c1-a820-8350d5399910-0', usage_metadata={'input_tokens': 1670, 'output_tokens': 75, 'total_tokens': 1745, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [41]:
out.content

'| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-05-26 00:00:00-04:00 | 430.26 | 435.2  | 426.12 |  433.59 | 4.56589e+07 |           0 |              0 |\n| 2026-05-27 00:00:00-04:00 | 442.89 | 445.6  | 435.52 |  440.36 | 4.46565e+07 |           0 |              0 |\n| 2026-05-28 00:00:00-04:00 | 437.62 | 443.96 | 436.3  |  442.1  | 3.2435e+07  |           0 |              0 |\n| 2026-05-29 00:00:00-04:00 | 439.85 | 441.07 | 428.14 |  435.79 | 4.51768e+07 |           0 |              0 |\n| 2026-06-01 00:00:00-04:00 | 427.49 | 429.6  | 415.43 |  415.88 | 4.49379e+07 |           0 |              0 |\n| 2026-06-02 00:00:00-04:00 | 418.22 | 424.42 | 413.65 |  423.74 | 3.7596e+07  |           0 |              0 |\n| 2026-06-03 00:00:00-04:00 | 418.7  | 433.6  | 416    |  423.7  | 4.45007e+07 |           0 | 

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

In [62]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
]


In [ ]:
from pdf_summary import summarize_pdf



# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [summarize_pdf]
tool_dict = {"summarize_pdf": summarize_pdf}

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = 'gpt-4o-mini')

llm_with_tools = llm.bind_tools(tools)



In [ ]:
tools

[<function pdf_summary.summarize_pdf(pdf_path)>]

In [63]:
# messages.append(messages)

response = llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[HumanMessage(content='이 c:\\Users\\Admin\\Desktop\\AI Autonomous\\cursor\\5일차\\samples/videoanomaly.pdf 문서의 저자는 누구야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_G1LZlfb3ac7LaN252bSzofLY', 'function': {'arguments': '{"pdf_path":"c:\\\\Users\\\\Admin\\\\Desktop\\\\AI Autonomous\\\\cursor\\\\5일차\\\\samples/videoanomaly.pdf"}', 'name': 'summarize_pdf'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 68, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DuvYvxljc9dcTLoJyolri2BkTsqYO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02e5-460e-

In [64]:
for tool_call in response.tool_calls:
    out = tool_dict[tool_call['name']].invoke(tool_call)
    messages.append(out)

llm_with_tools.invoke(messages)

AttributeError: 'function' object has no attribute 'invoke'

In [ ]:
## @tool 새로

from langchain_core.tools import tool
from pdf_summary import pdf_to_text, summarize_txt, summarize_pdf
@tool
def summarize_pdf_tool(pdf_path: str) -> str:
    """PDF 파일 경로를 받아 내용을 요약하고 요약 결과를 반환합니다.
    Args:
        pdf_path: PDF 파일의 전체 경로 (예: samples/Language_models.pdf)
    """
    return summarize_pdf(pdf_path)
@tool
def pdf_to_text_tool(pdf_path: str) -> str:
    """PDF 파일에서 텍스트를 추출해 txt 파일 경로를 반환합니다.
    Args:
        pdf_path: PDF 파일의 전체 경로
    """
    return pdf_to_text(pdf_path)
tools = [summarize_pdf_tool, pdf_to_text_tool]
llm_with_tools = llm.bind_tools(tools)